# UOB IA — Python Data Revision Guide
This notebook contains short, runnable examples and tiny exercises focused on practical data tasks for banking and operations.
Run cells sequentially in Jupyter Notebook or JupyterLab.

# Section 1: Python basics refresher

In [ ]:
# Variables and basic types
a = 10            # int
b = 3.14          # float
c = 'hello'       # string
d = True          # bool
print(type(a), type(b), type(c), type(d))
# Output example: <class 'int'> <class 'float'> <class 'str'> <class 'bool'>

In [ ]:
# Lists, dicts, tuples, sets
my_list = [1, 2, 3]
my_dict = {'agent_id': 'A01', 'calls': 12}
my_tuple = (10, 20)
my_set = {1,2,3}
print(my_list, my_dict, my_tuple, my_set)

## Loops, comprehensions and conditionals

In [ ]:
# for loop and list comprehension
squared = []
for x in range(5):
    squared.append(x * x)
squared2 = [x * x for x in range(5)]
print(squared, squared2)

# if / elif / else
score = 72
if score >= 80:
    grade = 'A'
elif score >= 60:
    grade = 'B'
else:
    grade = 'C'
print('grade =', grade)

## Functions and small exercises

In [ ]:
# function that returns average
def avg(numbers):
    return sum(numbers) / len(numbers) if numbers else None

print('avg durations:', avg([45, 30, 120, 60]))

# Exercise: filter names starting with A (solution below)
names = ['Alice', 'Bob', 'Andrew', 'Zoe']
a_names = [n for n in names if n.startswith('A')]
print('names starting with A:', a_names)

# Section 2: Pandas fundamentals (in-memory examples)

In [ ]:
import pandas as pd
import numpy as np

# Small contact-centre DataFrame
calls = pd.DataFrame([
    {'call_id': 'C001', 'timestamp':'2025-12-01 08:05:00', 'queue_time_sec': 10, 'handle_time_sec': 300, 'agent_id':'A01','resolved':True,'abandoned':False},
    {'call_id': 'C002', 'timestamp':'2025-12-01 08:12:00', 'queue_time_sec': 200, 'handle_time_sec': 0, 'agent_id':None,'resolved':False,'abandoned':True},
    {'call_id': 'C003', 'timestamp':'2025-12-01 09:01:00', 'queue_time_sec': 20, 'handle_time_sec': 180, 'agent_id':'A02','resolved':True,'abandoned':False},
    {'call_id': 'C004', 'timestamp':'2025-12-01 09:05:00', 'queue_time_sec': 120, 'handle_time_sec': 240, 'agent_id':'A01','resolved':False,'abandoned':False},
])
calls['timestamp'] = pd.to_datetime(calls['timestamp'])
calls['queue_time_sec'] = calls['queue_time_sec'].astype(int)
calls['handle_time_sec'] = calls['handle_time_sec'].astype(int)
calls['hour'] = calls['timestamp'].dt.hour
calls['weekday'] = calls['timestamp'].dt.day_name()
calls.head()

Exploring data: head(), info(), describe(), value_counts()

In [ ]:
calls.head()
print('INFO:')
calls.info()
print('DESCRIBE:')
print(calls.describe(include='all'))
print('agent counts:')
print(calls['agent_id'].value_counts(dropna=False))

Selecting columns/rows and boolean filters; handling missing data

In [ ]:
# select columns
print(calls[['call_id','queue_time_sec']])
# boolean filter
print(calls[calls['queue_time_sec'] > 60])
# missing values
print('missing counts:', calls.isna().sum())
print('fill missing agent_id:')
print(calls.fillna({'agent_id':'UNASSIGNED'})[['call_id','agent_id']])

# Section 3: Grouping, aggregation, KPIs

In [ ]:
# Abandonment and KPIs by hour
g = calls.groupby('hour').agg(total_calls=('call_id','count'), abandoned_calls=('abandoned', lambda x: x.sum()), avg_queue=('queue_time_sec','mean'))
g['abandon_rate'] = g['abandoned_calls'] / g['total_calls']
g.reset_index()

In [ ]:
# avg handle time by agent
by_agent = calls[~calls['abandoned']].groupby('agent_id').agg(handled_calls=('call_id','count'), avg_handle=('handle_time_sec','mean'))
by_agent.reset_index()

Exercise: calculate SLA breach rate (queue_time_sec > 120) by hour — add a boolean column and groupby.

In [ ]:
calls['sla_breach'] = calls['queue_time_sec'] > 120
sla = calls.groupby('hour').agg(total=('call_id','count'), breaches=('sla_breach','sum'))
sla['breach_rate'] = sla['breaches'] / sla['total']
sla.reset_index()

# Section 4: Joins and reshaping

In [ ]:
atm_master = pd.DataFrame([
    {'atm_id':'ATM01','location':'Orchard','capacity':200000,'last_topup':'2025-11-30'},
    {'atm_id':'ATM02','location':'Tampines','capacity':150000,'last_topup':'2025-11-29'},
])
transactions = pd.DataFrame([
    {'txn_id':'T001','atm_id':'ATM01','date':'2025-12-01','withdrawal_amount':200},
    {'txn_id':'T002','atm_id':'ATM01','date':'2025-12-01','withdrawal_amount':500},
    {'txn_id':'T003','atm_id':'ATM02','date':'2025-12-01','withdrawal_amount':1000},
])
tx = transactions.merge(atm_master, on='atm_id', how='left')
tx

In [ ]:
# pivot table example: sum withdrawal by location
pivot = tx.pivot_table(index='location', values='withdrawal_amount', aggfunc='sum')
pivot

# Section 5: Simple intelligent automation examples

In [ ]:
# Flag calls needing attention and ATM topup rule
calls['needs_attention'] = (calls['queue_time_sec'] > 120) | (calls['handle_time_sec'] == 0)
calls[['call_id','queue_time_sec','handle_time_sec','needs_attention']]

atm_tx_sum = tx.groupby('atm_id').withdrawal_amount.sum().reset_index().rename(columns={'withdrawal_amount':'sum_withdrawn'})
atm_status = atm_master.merge(atm_tx_sum, on='atm_id', how='left').fillna({'sum_withdrawn':0})
atm_status['topup_needed'] = atm_status['sum_withdrawn'] > (atm_status['capacity'] * 0.8)
atm_status[['atm_id','location','capacity','sum_withdrawn','topup_needed']]

In [ ]:
# Reusable function for flagging calls
def flag_calls_for_review(df, queue_threshold=120, zero_handle=True):
    df2 = df.copy()
    df2['flag'] = (df2['queue_time_sec'] > queue_threshold) | ((df2['handle_time_sec'] == 0) & zero_handle)
    return df2[df2['flag']]

flag_calls_for_review(calls, queue_threshold=60)

# Section 6: Optional light ML sketch (scikit-learn) — runs only if installed

In [ ]:
try:
    from sklearn.model_selection import train_test_split
    from sklearn.linear_model import LogisticRegression
    df_ml = calls.copy()
    df_ml['abandoned_int'] = df_ml['abandoned'].astype(int)
    X = df_ml[['queue_time_sec','hour']]
    y = df_ml['abandoned_int']
    X_train, X_test, y_train, y_test = train_test_split(X,y,random_state=42,test_size=0.4)
    model = LogisticRegression().fit(X_train,y_train)
    print('accuracy', model.score(X_test,y_test))
except Exception as e:
    print('scikit-learn not available or error:', e)

# Appendix: Case management example and KPIs

In [ ]:
cases = pd.DataFrame([
    {'case_id':'CA001','created_time':'2025-11-20 09:00','closed_time':'2025-11-20 12:00','status':'Closed','team':'Onboarding','priority':'High'},
    {'case_id':'CA002','created_time':'2025-11-21 10:00','closed_time':None,'status':'Open','team':'Onboarding','priority':'Low'},
    {'case_id':'CA003','created_time':'2025-11-19 08:00','closed_time':'2025-11-19 09:30','status':'Closed','team':'Support','priority':'Medium'},
])
cases['created_time'] = pd.to_datetime(cases['created_time'])
cases['closed_time'] = pd.to_datetime(cases['closed_time'])
cases['resolution_hours'] = (cases['closed_time'] - cases['created_time']).dt.total_seconds() / 3600
cases[['case_id','team','status','resolution_hours']]

# KPI: avg resolution by team
cases.groupby('team').resolution_hours.mean()

---
End of guided notebook. Modify thresholds and data to practice further.